In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip "/content/drive/MyDrive/images_splited.zip" -d /content/data


Archive:  /content/drive/MyDrive/images_splited.zip
   creating: /content/data/images_splited/
   creating: /content/data/images_splited/test/
   creating: /content/data/images_splited/train/
   creating: /content/data/images_splited/test/Сyrestis_thyodamas/
   creating: /content/data/images_splited/test/Hestina_assimilis_assimilis/
   creating: /content/data/images_splited/test/Polygonia_c-aureum/
   creating: /content/data/images_splited/test/Argynnis_hyperbius/
   creating: /content/data/images_splited/test/Graphium_sarpedon_nipponum/
   creating: /content/data/images_splited/test/Cephonodes_hylas/
   creating: /content/data/images_splited/test/Papilio_xuthus.csv/
   creating: /content/data/images_splited/test/Lampides_boeticus/
   creating: /content/data/images_splited/test/Eurema_mandarina/
   creating: /content/data/images_splited/test/Pseudozizeeria_maha/
   creating: /content/data/images_splited/train/Сyrestis_thyodamas/
   creating: /content/data/images_splited/train/Hestina_a

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils import data
from torchvision import transforms, models, datasets
from PIL import Image
import json
import os
from tqdm import tqdm
import numpy as np


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device

device(type='cuda')

In [ ]:

data_path = '/content/data/images_splited'
train_path = os.path.join(data_path, 'train')
test_path = os.path.join(data_path, 'test')

In [ ]:
resnet_weights = models.ResNet50_Weights.DEFAULT
resnet_transforms = resnet_weights.transforms()

test_transform = resnet_transforms

train_transform = transforms.Compose([
        transforms.RandomResizedCrop(244),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])



Иницилизация датасета c помощью ImageFolder и Создание Dataloader(специальный класс для загрузки данных ввиде батча)

In [ ]:
train_dataset = datasets.ImageFolder(root=train_path,transform=train_transform)

test_dataset = datasets.ImageFolder(root=test_path,transform=test_transform)

train_loader = data.DataLoader(train_dataset,batch_size=32, shuffle=True, num_workers=2)

test_loader = data.DataLoader(test_dataset,batch_size=32,shuffle=False, num_workers=2)


In [ ]:
train_dataset.class_to_idx

{'Argynnis_hyperbius': 0,
 'Cephonodes_hylas': 1,
 'Eurema_mandarina': 2,
 'Graphium_sarpedon_nipponum': 3,
 'Hestina_assimilis_assimilis': 4,
 'Lampides_boeticus': 5,
 'Papilio_xuthus.csv': 6,
 'Polygonia_c-aureum': 7,
 'Pseudozizeeria_maha': 8,
 'Сyrestis_thyodamas': 9}

In [ ]:
model.fc = nn.Sequential(
    nn.Linear(2048, 512),       # Дополнительный полносвязный слой
    nn.ReLU(),                  # Активация
    nn.Dropout(0.5),            # Отключение
    nn.Linear(512, 10)          # Выходной слой (10 классов)
)


model.fc.requires_grad_(True)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

model.fc = nn.Linear(512*4, 10)
model.fc.requires_grad_(True)

In [ ]:
targets = train_dataset.targets
# Считаем количество примеров для каждого класса
counts = np.bincount(targets)
# Вычисляем веса: (1 / кол-во), чтобы редкие классы имели больший вес
# Можно умножить на counts.sum(), чтобы числа не были слишком маленькими, но это не обязательно для Softmax
weights = 1. / counts
# Переводим в тензор и отправляем на то же устройство, где модель (GPU)
class_weights = torch.FloatTensor(weights).to(device)

# --- 2. НАСТРОЙКА ОБУЧЕНИЯ ---
optimizer = optim.SGD(params=model.fc.parameters(), lr=0.01, momentum=0.9, weight_decay=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ВАЖНО: передаем рассчитанные веса в функцию потерь
loss_function = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
model = model.to(device)
model.train()
epochs = 20

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss = 0.9 * running_loss + 0.1 * loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()

    accuracy = 100 * correct / total
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1}, Loss: {running_loss:.4f}, Accuracy: {accuracy:.2f}%, LR: {current_lr}')

Epoch 1, Loss: 2.1011, Accuracy: 26.97%, LR: 0.01
Epoch 2, Loss: 1.5971, Accuracy: 53.34%, LR: 0.01
Epoch 3, Loss: 1.0475, Accuracy: 67.13%, LR: 0.01
Epoch 4, Loss: 0.8467, Accuracy: 75.56%, LR: 0.01
Epoch 5, Loss: 0.7464, Accuracy: 78.81%, LR: 0.01
Epoch 6, Loss: 0.6581, Accuracy: 79.89%, LR: 0.01
Epoch 7, Loss: 0.5483, Accuracy: 82.00%, LR: 0.001
Epoch 8, Loss: 0.5165, Accuracy: 83.74%, LR: 0.001
Epoch 9, Loss: 0.4981, Accuracy: 84.77%, LR: 0.001
Epoch 10, Loss: 0.4973, Accuracy: 84.29%, LR: 0.001
Epoch 11, Loss: 0.5068, Accuracy: 84.95%, LR: 0.001
Epoch 12, Loss: 0.4927, Accuracy: 85.61%, LR: 0.001
Epoch 13, Loss: 0.5589, Accuracy: 84.35%, LR: 0.001
Epoch 14, Loss: 0.4538, Accuracy: 86.94%, LR: 0.0001
Epoch 15, Loss: 0.4877, Accuracy: 84.59%, LR: 0.0001
Epoch 16, Loss: 0.4824, Accuracy: 85.37%, LR: 0.0001
Epoch 17, Loss: 0.5185, Accuracy: 85.91%, LR: 0.0001
Epoch 18, Loss: 0.5016, Accuracy: 86.33%, LR: 0.0001
Epoch 19, Loss: 0.4503, Accuracy: 85.97%, LR: 0.0001
Epoch 20, Loss: 0.417

In [ ]:
st = model.state_dict()
torch.save(st, 'model_weightss_butterfly1.tar')

In [ ]:
# Тестирование
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 92.36%
